<a href="https://colab.research.google.com/github/thbieltraining/Calculadora-SENAI/blob/main/CALCULADORA_SENAI_ADME2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:








import re
from decimal import Decimal, ROUND_HALF_UP


# ================================================================
# CALCULADORA FOPAG SENAI 2026
# Motor de cálculo - versão base para Django / Tkinter
# ================================================================


# ================================================================
# CONFIGURAÇÕES
# ================================================================

SALARIO_MINIMO_2026 = Decimal("1621.00")
DEDUCAO_DEPENDENTE_IRRF = Decimal("189.59")
DEDUCAO_SIMPLIFICADA_IRRF_2026 = Decimal("607.20")
TETO_INSS = Decimal("988.09")

DOIS_CASAS = Decimal("0.01")


# ================================================================
# TABELAS 2026 - CONFORME ESPECIFICAÇÃO DO PROJETO
# ================================================================

TABELA_INSS = [
    (Decimal("1621.00"), Decimal("0.075"), Decimal("0.00")),
    (Decimal("2902.84"), Decimal("0.09"), Decimal("24.31")),
    (Decimal("4354.27"), Decimal("0.12"), Decimal("111.40")),
    (Decimal("8475.55"), Decimal("0.14"), Decimal("198.49")),
]


TABELA_IRRF = [
    (Decimal("2428.80"), Decimal("0.00"), Decimal("0.00")),
    (Decimal("2826.65"), Decimal("0.075"), Decimal("182.16")),
    (Decimal("3751.05"), Decimal("0.15"), Decimal("394.16")),
    (Decimal("4664.68"), Decimal("0.225"), Decimal("675.49")),
    (Decimal("999999999999.99"), Decimal("0.275"), Decimal("908.73")),
]


NOMES_MESES = [
    "",
    "Janeiro",
    "Fevereiro",
    "Março",
    "Abril",
    "Maio",
    "Junho",
    "Julho",
    "Agosto",
    "Setembro",
    "Outubro",
    "Novembro",
    "Dezembro"
]


# ================================================================
# FUNÇÕES BÁSICAS
# ================================================================

def D(valor):
    """
    Converte qualquer valor numérico para Decimal
    de maneira segura.
    """
    return Decimal(str(valor))


def arredonda(valor):
    """
    Arredondamento financeiro SENAI:
    Half-Up em 2 casas decimais.
    """
    return D(valor).quantize(DOIS_CASAS, rounding=ROUND_HALF_UP)


def dinheiro(valor):
    """
    Formatação monetária para exibição.
    """
    return f"R$ {arredonda(valor):.2f}"


def ler_numero(mensagem, valor_padrao, tipo=Decimal):
    """
    Entrada numérica com valor padrão.
    Aceita vírgula ou ponto.
    """
    while True:
        entrada = input(f"{mensagem} [{valor_padrao}]: ").strip()

        if entrada == "":
            return tipo(str(valor_padrao).replace(",", "."))

        try:
            return tipo(entrada.replace(",", "."))
        except ValueError:
            print("⚠️ Valor inválido. Digite novamente.")


def ler_sim_nao(mensagem, padrao="n"):
    """
    Entrada simples de sim/não.
    """
    while True:
        entrada = input(f"{mensagem} (s/n) [{padrao}]: ").strip().lower()

        if entrada == "":
            entrada = padrao

        if entrada in ("s", "sim"):
            return True

        if entrada in ("n", "nao", "não"):
            return False

        print("⚠️ Informe apenas S ou N.")


def validar_mes(mes):
    if mes < 1 or mes > 12:
        raise ValueError("O mês deve estar entre 1 e 12.")


def validar_ano(ano):
    if ano < 2000:
        raise ValueError("Ano inválido.")


def capturar_mes_ano(mensagem, mes_padrao, ano_padrao):
    """Lê período no formato M-AAAA (ex.: 2-2026)."""
    while True:
        entrada = input(
            f"{mensagem} [{mes_padrao}-{ano_padrao}]: "
        ).strip()

        if not entrada:
            return mes_padrao, ano_padrao

        match = re.fullmatch(r"(\d{1,2})-(\d{4})", entrada)

        if not match:
            print("⚠️ Informe no formato M-AAAA. Ex.: 2-2026")
            continue

        mes = int(match.group(1))
        ano = int(match.group(2))

        try:
            validar_mes(mes)
            validar_ano(ano)
            return mes, ano
        except ValueError as erro:
            print(f"⚠️ {erro}")


def proximo_mes(mes, ano):
    """Retorna o mês seguinte, tratando a virada do ano."""
    if mes == 12:
        return 1, ano + 1
    return mes + 1, ano


# ================================================================
# INSS
# ================================================================

def calc_inss(base):
    """
    Calcula o INSS conforme a tabela progressiva definida
    no projeto.

    O resultado é limitado ao teto de R$ 989,08.
    """

    base = arredonda(base)

    if base <= 0:
        return Decimal("0.00")

    for teto, aliquota, deducao in TABELA_INSS:

        if base <= teto:

            valor = arredonda(
                base * aliquota - deducao
            )

            return min(
                max(valor, Decimal("0.00")),
                TETO_INSS
            )

    # Acima do teto da última faixa
    valor = arredonda(
        TABELA_INSS[-1][0] *
        TABELA_INSS[-1][1]
        - TABELA_INSS[-1][2]
    )

    return min(valor, TETO_INSS)


# ================================================================
# IRRF
# ================================================================

def _irrf_progressivo(base):
    """Calcula o IRRF pela tabela progressiva vigente em 2026."""
    base = arredonda(base)
    if base <= 0:
        return Decimal("0.00")

    for teto, aliquota, deducao in TABELA_IRRF:
        if base <= teto:
            return max(arredonda(base * aliquota - deducao), Decimal("0.00"))
    return Decimal("0.00")


def _reducao_irrf_2026(rendimento_tributavel):
    rendimento_tributavel = arredonda(rendimento_tributavel)
    if rendimento_tributavel <= Decimal("5000.00"):
        return min(_irrf_progressivo(rendimento_tributavel), Decimal("312.89"))
    if rendimento_tributavel <= Decimal("7350.00"):
        reducao = Decimal("978.62") - (Decimal("0.133145") * rendimento_tributavel)
        return max(arredonda(reducao), Decimal("0.00"))
    return Decimal("0.00")


def calc_irrf(base_legal, bruto_total_tributavel):
    """Calcula IRRF 2026 comparando dedução legal e simplificada."""
    base_legal = arredonda(base_legal)
    bruto_total_tributavel = arredonda(bruto_total_tributavel)
    if base_legal <= 0:
        return Decimal("0.00")

    base_simplificada = max(
        arredonda(bruto_total_tributavel - DEDUCAO_SIMPLIFICADA_IRRF_2026),
        Decimal("0.00")
    )
    imposto_legal = _irrf_progressivo(base_legal)
    imposto_simplificado = _irrf_progressivo(base_simplificada)
    imposto = min(imposto_legal, imposto_simplificado)
    reducao = _reducao_irrf_2026(bruto_total_tributavel)
    return max(arredonda(imposto - reducao), Decimal("0.00"))


# ================================================================
# ADICIONAIS DE RISCO
# ================================================================

def capturar_adicionais_risco(salario_base):

    tem_adicional = ler_sim_nao(
        "Possui adicional de Insalubridade ou Periculosidade?",
        "n"
    )

    if not tem_adicional:
        return (
            Decimal("0.00"),
            "Nenhum",
            Decimal("0.00")
        )

    while True:

        tipo = input(
            "Qual o adicional? "
            "(i = Insalubridade, p = Periculosidade) [p]: "
        ).strip().lower()

        if tipo == "":
            tipo = "p"

        if tipo == "p":

            valor = arredonda(
                salario_base * Decimal("0.30")
            )

            return (
                valor,
                "Periculosidade (30%)",
                Decimal("0.30")
            )

        if tipo == "i":

            print("\nGraus de Insalubridade:")
            print("1 = Mínimo (10%)")
            print("2 = Médio  (20%)")
            print("3 = Máximo (40%)")

            while True:

                grau = input(
                    "Informe o grau (1, 2 ou 3) [2]: "
                ).strip()

                if grau == "":
                    grau = "2"

                if grau in ("1", "2", "3"):
                    grau = int(grau)
                    break

                print("⚠️ Grau inválido.")

            percentuais = {
                1: Decimal("0.10"),
                2: Decimal("0.20"),
                3: Decimal("0.40")
            }

            percentual = percentuais[grau]

            valor = arredonda(
                SALARIO_MINIMO_2026 * percentual
            )

            return (
                valor,
                f"Insalubridade ({percentual * 100:.0f}%)",
                percentual
            )

        print("⚠️ Informe I ou P.")


# ================================================================
# HORA BASE
# ================================================================

def calcular_hora_base(remuneracao_base, carga_horaria):

    if carga_horaria <= 0:
        raise ValueError(
            "A carga horária deve ser maior que zero."
        )

    return arredonda(
        remuneracao_base / carga_horaria
    )


# ================================================================
# HORAS EXTRAS
# ================================================================

def calcular_he_duas_etapas(
    hora_base,
    multiplicador,
    quantidade_horas
):

    if quantidade_horas <= 0:
        return (
            Decimal("0.00"),
            Decimal("0.00")
        )

    # Etapa 1:
    # calcula e arredonda o valor unitário
    valor_unitario = arredonda(
        hora_base * multiplicador
    )

    # Etapa 2:
    # multiplica o valor já arredondado
    total = arredonda(
        valor_unitario * quantidade_horas
    )

    return valor_unitario, total


# ================================================================
# FALTAS
# ================================================================

def capturar_faltas_multiplos_meses(mes_vigente):

    houve_faltas = ler_sim_nao(
        "Houve faltas injustificáveis?",
        "n"
    )

    if not houve_faltas:
        return (
            0,
            0,
            {},
            0,
            0
        )

    print("\n[FALTAS - FORMATO MÊS-SEMANA-DIAS]")
    print("Exemplo:")
    print("4-3-1, 5-1-1, 9-2-2")

    entrada = input(
        f"Informe as faltas [ex: {mes_vigente}-1-1]: "
    ).strip()

    if not entrada:
        entrada = f"{mes_vigente}-1-1"

    padrao = r"(\d{1,2})\s*-\s*([1-4])\s*-\s*(\d+)"

    ocorrencias = re.findall(padrao, entrada)

    if not ocorrencias:
        print("⚠️ Nenhuma falta válida identificada.")
        return 0, 0, {}, 0, 0

    faltas_por_mes = {}

    for mes, semana, dias in ocorrencias:

        mes = int(mes)
        semana = int(semana)
        dias = int(dias)

        if mes < 1 or mes > 12:
            print(
                f"⚠️ Mês {mes} ignorado: mês inválido."
            )
            continue

        if dias <= 0:
            continue

        if mes not in faltas_por_mes:
            faltas_por_mes[mes] = {
                1: 0,
                2: 0,
                3: 0,
                4: 0,
                "total_dias": 0
            }

        dados = faltas_por_mes[mes]

        # Não há limite mensal de 14 faltas. O limite para férias é
        # apurado sobre o total de faltas injustificadas no período aquisitivo.
        dias_aplicados = dias

        dados[semana] += dias_aplicados
        dados["total_dias"] += dias_aplicados

    # Total geral informado
    total_dias_global = sum(
        dados["total_dias"]
        for dados in faltas_por_mes.values()
    )

    # Semanas com DSR perdido - todos os meses
    total_semanas_dsr_global = sum(
        sum(
            1
            for semana in range(1, 5)
            if dados[semana] > 0
        )
        for dados in faltas_por_mes.values()
    )

    # Apenas competência atual
    dados_vigente = faltas_por_mes.get(
        mes_vigente,
        {}
    )

    dias_mes_vigente = dados_vigente.get(
        "total_dias",
        0
    )

    semanas_dsr_vigente = sum(
        1
        for semana in range(1, 5)
        if dados_vigente.get(semana, 0) > 0
    )

    return (
        total_dias_global,
        total_semanas_dsr_global,
        faltas_por_mes,
        dias_mes_vigente,
        semanas_dsr_vigente
    )


def calcular_desconto_faltas(
    salario_base,
    valor_adicional,
    dias_faltados,
    semanas_perda_dsr,
    media_variaveis=Decimal("0.00")
):

    if dias_faltados <= 0:
        return (
            Decimal("0.00"),
            Decimal("0.00"),
            Decimal("0.00"),
            Decimal("0.00")
        )

    remuneracao_base = arredonda(
        salario_base + valor_adicional
    )

    # Valor do dia conforme especificação
    remuneracao_ferias = arredonda(remuneracao_base + media_variaveis)

    valor_dia = arredonda(
        remuneracao_ferias / Decimal("30")
    )

    desconto_faltas = arredonda(
        valor_dia * dias_faltados
    )

    desconto_dsr = arredonda(
        valor_dia * semanas_perda_dsr
    )

    total = arredonda(
        desconto_faltas + desconto_dsr
    )

    return (
        valor_dia,
        desconto_faltas,
        desconto_dsr,
        total
    )


# ================================================================
# FÉRIAS
# ================================================================

def dias_ferias_por_faltas(total_faltas_periodo_aquisitivo):
    """Aplica a tabela do art. 130 da CLT."""
    faltas = int(total_faltas_periodo_aquisitivo)
    if faltas <= 5:
        return 30
    if faltas <= 14:
        return 24
    if faltas <= 23:
        return 18
    if faltas <= 32:
        return 12
    return 0


def capturar_media_variaveis_12_meses(nome_verba):
    """Captura a média dos valores variáveis dos últimos 12 meses."""
    usar_media = ler_sim_nao(
        f"Deseja informar a média de 12 meses de {nome_verba}?", "n"
    )
    if not usar_media:
        return Decimal("0.00"), []

    valores = []
    for mes in range(1, 13):
        valor = ler_numero(
            f"{nome_verba} - {NOMES_MESES[mes]} (R$)", "0.00"
        )
        if valor < 0:
            raise ValueError(f"Valor negativo não permitido para {nome_verba}.")
        valores.append(arredonda(valor))

    media = arredonda(sum(valores, Decimal("0.00")) / Decimal("12"))
    return media, valores


def capturar_e_calcular_ferias(
    remuneracao_base,
    mes_competencia,
    ano_competencia,
    faltas_periodo_aquisitivo=0,
    media_variaveis=Decimal("0.00")
):

    quer_ferias = ler_sim_nao(
        "Deseja programar/calcular Férias?",
        "n"
    )

    if not quer_ferias:
        return None

    entrada_periodo = input(
        "Quando as férias serão tiradas? "
        "('mesmo' ou ex: '2-2027') [mesmo]: "
    ).strip().lower()

    if not entrada_periodo:
        entrada_periodo = "mesmo"

    mes_gozo = mes_competencia
    ano_gozo = ano_competencia

    if entrada_periodo != "mesmo":

        match = re.fullmatch(
            r"(\d{1,2})-(\d{4})",
            entrada_periodo
        )

        if not match:
            print(
                "⚠️ Período inválido. "
                "Considerando mês da competência."
            )
        else:
            mes_gozo = int(match.group(1))
            ano_gozo = int(match.group(2))

            validar_mes(mes_gozo)
            validar_ano(ano_gozo)

    dias_maximos = dias_ferias_por_faltas(faltas_periodo_aquisitivo)
    if dias_maximos == 0:
        raise ValueError(
            "Mais de 32 faltas injustificadas no período aquisitivo: "
            "não há direito às férias pelo art. 130 da CLT."
        )

    print(
        f"Direito de férias pelo art. 130 da CLT: {dias_maximos} dias "
        f"({faltas_periodo_aquisitivo} faltas injustificadas)."
    )

    dias_gozo = ler_numero(
        f"Dias de Férias gozados (máx. {dias_maximos})",
        dias_maximos,
        int
    )

    if dias_gozo <= 0 or dias_gozo > dias_maximos:
        raise ValueError(
            "Dias de férias devem estar entre 1 e 30."
        )

    vender_dez_dias = ler_sim_nao(
        "Houve abono pecuniário (vendeu 10 dias)?",
        "n"
    )

    # Remuneração usada como base das férias.
    # Se não houver média de variáveis, ela já vem como R$ 0,00.
    remuneracao_ferias = arredonda(
        remuneracao_base + media_variaveis
    )

    valor_dia = arredonda(
        remuneracao_ferias / Decimal("30")
    )

    # Férias gozadas
    valor_ferias = arredonda(
        valor_dia * dias_gozo
    )

    # 1/3 constitucional
    terco_ferias = arredonda(
        valor_ferias / Decimal("3")
    )

    total_tributavel = arredonda(
        valor_ferias + terco_ferias
    )

    # Abono
    valor_abono = Decimal("0.00")
    terco_abono = Decimal("0.00")
    total_isento = Decimal("0.00")

    if vender_dez_dias:

        valor_abono = arredonda(
            valor_dia * Decimal("10")
        )

        terco_abono = arredonda(
            valor_abono / Decimal("3")
        )

        total_isento = arredonda(
            valor_abono + terco_abono
        )

    total_bruto = arredonda(
        total_tributavel + total_isento
    )

    eh_mes_vigente = (
        mes_gozo == mes_competencia
        and ano_gozo == ano_competencia
    )

    return {
        "mes_gozo": mes_gozo,
        "ano_gozo": ano_gozo,
        "eh_mes_vigente": eh_mes_vigente,
        "dias_gozo": dias_gozo,

        "faltas_periodo_aquisitivo": faltas_periodo_aquisitivo,
        "dias_direito": dias_maximos,
        "media_variaveis": media_variaveis,
        "remuneracao_ferias": remuneracao_ferias,
        "valor_dia": valor_dia,

        "valor_ferias": valor_ferias,
        "terco_ferias": terco_ferias,
        "total_tributavel": total_tributavel,

        "vendeu_abono": vender_dez_dias,
        "valor_abono": valor_abono,
        "terco_abono": terco_abono,
        "total_isento": total_isento,

        "total_bruto": total_bruto
    }


# ================================================================
# 13º SALÁRIO
# ================================================================

def capturar_e_calcular_13(
    remuneracao_base,
    total_he,
    dependentes,
    mes_competencia,
    ano_competencia,
    media_variaveis=Decimal("0.00")
):

    quer_13 = ler_sim_nao(
        "Deseja calcular 13º Salário?",
        "n"
    )

    if not quer_13:
        return None

    entrada_periodo = input(
        f"Em qual mês/ano o 13º será pago? "
        f"[12-{ano_competencia}]: "
    ).strip().lower()

    if not entrada_periodo:
        entrada_periodo = f"12-{ano_competencia}"

    mes_pago = 12
    ano_pago = ano_competencia

    if entrada_periodo != "mesmo":

        match = re.fullmatch(
            r"(\d{1,2})-(\d{4})",
            entrada_periodo
        )

        if not match:
            raise ValueError(
                "Período do 13º inválido."
            )

        mes_pago = int(match.group(1))
        ano_pago = int(match.group(2))

        validar_mes(mes_pago)
        validar_ano(ano_pago)

    else:
        mes_pago = mes_competencia

    opcao = input(
        "Qual a modalidade? "
        "[1=1ª Parcela, 2=2ª Parcela, d=Duas Parcelas] [2]: "
    ).strip().lower()

    if not opcao:
        opcao = "2"

    if opcao not in ("1", "2", "d"):
        raise ValueError(
            "Modalidade de 13º inválida."
        )

    avos = ler_numero(
        "Número de avos/meses trabalhados no ano (1 a 12)",
        12,
        int
    )

    if avos < 1 or avos > 12:
        raise ValueError(
            "Os avos devem estar entre 1 e 12."
        )

    # ------------------------------------------------------------
    # BASE FIXA PROPORCIONAL
    # ------------------------------------------------------------

    base_fixa_13 = arredonda(
        (remuneracao_base / Decimal("12"))
        * Decimal(avos)
    )

    # 1ª parcela:
    # 50% somente da verba fixa proporcional
    primeira_parcela = arredonda(
        base_fixa_13 * Decimal("0.50")
    )

    # ------------------------------------------------------------
    # BASE TOTAL COM MÉDIA DE HORAS EXTRAS
    # ------------------------------------------------------------

    media_variavel_13 = arredonda(
        media_variaveis if media_variaveis > 0 else total_he
    )

    base_total_13 = arredonda(
        (remuneracao_base + media_variavel_13)
        / Decimal("12")
        * Decimal(avos)
    )

    # ------------------------------------------------------------
    # INSS DO 13º
    # ------------------------------------------------------------

    inss_13 = calc_inss(
        base_total_13
    )

    # ------------------------------------------------------------
    # IRRF DO 13º
    # ------------------------------------------------------------

    base_irrf_13 = arredonda(
        base_total_13
        - inss_13
        - (
            Decimal(dependentes)
            * DEDUCAO_DEPENDENTE_IRRF
        )
    )

    irrf_13 = calc_irrf(
        base_irrf_13,
        base_total_13
    )

    # ------------------------------------------------------------
    # 2ª PARCELA
    # ------------------------------------------------------------

    segunda_parcela = arredonda(
        base_total_13
        - primeira_parcela
        - inss_13
        - irrf_13
    )

    eh_mes_vigente = (
        mes_pago == mes_competencia
        and ano_pago == ano_competencia
    )

    return {
        "mes_pago": mes_pago,
        "ano_pago": ano_pago,
        "eh_mes_vigente": eh_mes_vigente,

        "opcao": opcao,
        "avos": avos,

        "base_fixa_13": base_fixa_13,
        "media_variavel_13": media_variavel_13,
        "base_total_13": base_total_13,

        "primeira_parcela": primeira_parcela,

        "inss_13": inss_13,
        "base_irrf_13": base_irrf_13,
        "irrf_13": irrf_13,

        "segunda_parcela": segunda_parcela
    }


# ================================================================
# RESCISÃO
# ================================================================

from enum import Enum


class MotivoRescisao(Enum):
    SEM_JUSTA_CAUSA = "sem_justa_causa"
    PEDIDO_DEMISSAO = "pedido_demissao"
    JUSTA_CAUSA = "justa_causa"
    RESCISAO_INDIRETA = "rescisao_indireta"
    TERMINO_EXPERIENCIA = "termino_experiencia"
    ANTECIPACAO_EXPERIENCIA_EMPREGADOR = "antecipacao_experiencia_empregador"
    ANTECIPACAO_EXPERIENCIA_EMPREGADO = "antecipacao_experiencia_empregado"


def calcular_rescisao_experiencia(salario_base, dias_restantes, motivo):
    """Calcula a indenização do art. 479 quando o empregador rompe antes do termo."""
    salario_base = arredonda(salario_base)
    dias_restantes = max(int(dias_restantes), 0)
    valor_dia = arredonda(salario_base / Decimal("30"))

    indenizacao = Decimal("0.00")
    if motivo == MotivoRescisao.ANTECIPACAO_EXPERIENCIA_EMPREGADOR:
        indenizacao = arredonda(valor_dia * Decimal(dias_restantes) / Decimal("2"))

    return {
        "motivo": motivo.value,
        "dias_restantes": dias_restantes,
        "indenizacao_art_479": indenizacao,
    }


def calcular_aviso_previo_com_media(salario_base, dias_aviso, media_variaveis=Decimal("0.00")):
    """Base do aviso prévio considerando remuneração fixa + média variável."""
    remuneracao_media = arredonda(salario_base + media_variaveis)
    valor_dia = arredonda(remuneracao_media / Decimal("30"))
    return {
        "remuneracao_media": remuneracao_media,
        "dias_aviso": int(dias_aviso),
        "valor_aviso": arredonda(valor_dia * Decimal(int(dias_aviso))),
    }


def calcular_multa_art_477(salario_base, pagamento_no_prazo, multa_aplicavel=True):
    """Evento separado para a multa do art. 477, sem presumir sua incidência."""
    if pagamento_no_prazo or not multa_aplicavel:
        return Decimal("0.00")
    return arredonda(salario_base)


def identificar_direitos_rescisao(motivo):
    """Mapa inicial de eventos; o cálculo definitivo depende do caso concreto."""
    return {
        "motivo": motivo.value,
        "saldo_salario": True,
        "decimo_terceiro": motivo != MotivoRescisao.JUSTA_CAUSA,
        "ferias_vencidas": True,
        "ferias_proporcionais": motivo not in (MotivoRescisao.JUSTA_CAUSA,),
        "aviso_previo": motivo in (MotivoRescisao.SEM_JUSTA_CAUSA, MotivoRescisao.RESCISAO_INDIRETA),
    }


# ================================================================
# EXECUÇÃO
# ================================================================

def executar_calculadora():

    print("=" * 70)
    print("        CALCULADORA FOPAG SENAI 2026")
    print("=" * 70)

    # ------------------------------------------------------------
    # 1. SALÁRIO
    # ------------------------------------------------------------

    salario = ler_numero(
        "Digite o Salário Fixo (R$)",
        "2950.00"
    )

    if salario <= 0:
        raise ValueError(
            "O salário deve ser maior que zero."
        )

    # ------------------------------------------------------------
    # 2. ADICIONAIS
    # ------------------------------------------------------------

    valor_adicional, desc_adicional, perc_adicional = (
        capturar_adicionais_risco(salario)
    )

    # ------------------------------------------------------------
    # 3. CARGA HORÁRIA
    # ------------------------------------------------------------

    carga_horaria = ler_numero(
        "Carga Horária Mensal em Horas (ex: 220, 200)",
        "220.00"
    )

    if carga_horaria <= 0:
        raise ValueError(
            "Carga horária inválida."
        )

    # ------------------------------------------------------------
    # 4. COMPETÊNCIA
    # ------------------------------------------------------------

    mes_competencia = ler_numero(
        "Mês Vigente da Folha (1 a 12)",
        9,
        int
    )

    validar_mes(mes_competencia)

    ano_competencia = 2026

    # ------------------------------------------------------------
    # 4.1. RECEBIMENTO DO SALÁRIO
    # ------------------------------------------------------------

    mes_pagamento_salario, ano_pagamento_salario = proximo_mes(
        mes_competencia,
        ano_competencia
    )

    mes_pagamento_salario, ano_pagamento_salario = capturar_mes_ano(
        "Em qual mês/ano o salário desta competência será recebido?",
        mes_pagamento_salario,
        ano_pagamento_salario
    )

    print("-" * 70)

    # ------------------------------------------------------------
    # 5. FALTAS
    # ------------------------------------------------------------

    (
        dias_global,
        semanas_global,
        mapa_faltas,
        dias_vigente,
        semanas_vigente
    ) = capturar_faltas_multiplos_meses(
        mes_competencia
    )

    print("-" * 70)

    # ------------------------------------------------------------
    # 6. HORAS EXTRAS
    # ------------------------------------------------------------

    he50_qtd = ler_numero(
        "Horas Extras 50% (quantidade)",
        "18.00"
    )

    he70_qtd = ler_numero(
        "Horas Extras 70% (quantidade)",
        "0.00"
    )

    he100_qtd = ler_numero(
        "Horas Extras 100% (quantidade)",
        "10.00"
    )

    dependentes = ler_numero(
        "Número de Dependentes",
        1,
        int
    )

    if any(qtd < 0 for qtd in (
        he50_qtd,
        he70_qtd,
        he100_qtd
    )):
        raise ValueError(
            "Quantidade de horas extras não pode ser negativa."
        )

    if dependentes < 0:
        raise ValueError(
            "Número de dependentes não pode ser negativo."
        )

    # ------------------------------------------------------------
    # 7. REMUNERAÇÃO BASE
    # ------------------------------------------------------------

    remuneracao_base = arredonda(
        salario + valor_adicional
    )

    hora_base = calcular_hora_base(
        remuneracao_base,
        carga_horaria
    )

    # ------------------------------------------------------------
    # 8. HORAS EXTRAS
    # ------------------------------------------------------------

    u_he50, v_he50 = calcular_he_duas_etapas(
        hora_base,
        Decimal("1.50"),
        he50_qtd
    )

    u_he70, v_he70 = calcular_he_duas_etapas(
        hora_base,
        Decimal("1.70"),
        he70_qtd
    )

    u_he100, v_he100 = calcular_he_duas_etapas(
        hora_base,
        Decimal("2.00"),
        he100_qtd
    )

    total_he = arredonda(
        v_he50
        + v_he70
        + v_he100
    )

    # ------------------------------------------------------------
    # 9. MÉDIA DE VARIÁVEIS
    # ------------------------------------------------------------

    media_variaveis, historico_variaveis = capturar_media_variaveis_12_meses(
        "verbas variáveis"
    )

    print("-" * 70)

    # ------------------------------------------------------------
    # 10. FÉRIAS
    # ------------------------------------------------------------

    dados_ferias = capturar_e_calcular_ferias(
        remuneracao_base,
        mes_competencia,
        ano_competencia,
        dias_global,
        media_variaveis
    )

    print("-" * 70)

    # ------------------------------------------------------------
    # 10. 13º
    # ------------------------------------------------------------

    dados_13 = capturar_e_calcular_13(
        remuneracao_base,
        total_he,
        dependentes,
        mes_competencia,
        ano_competencia,
        media_variaveis
    )

    print("-" * 70)

    # ------------------------------------------------------------
    # 11. VT
    # ------------------------------------------------------------

    valor_vt_total = ler_numero(
        "Valor TOTAL do VT gasto no mês (R$)",
        "180.00"
    )

    if valor_vt_total < 0:
        raise ValueError(
            "Valor do VT não pode ser negativo."
        )

    # ------------------------------------------------------------
    # 12. FALTAS DA COMPETÊNCIA
    # ------------------------------------------------------------

    (
        valor_dia,
        desconto_faltas,
        desconto_dsr,
        total_desconto_faltas
    ) = calcular_desconto_faltas(
        salario,
        valor_adicional,
        dias_vigente,
        semanas_vigente,
        media_variaveis
    )

    # ------------------------------------------------------------
    # 13. FOLHA MENSAL
    # ------------------------------------------------------------

    # A competência é o mês trabalhado/folha fechada.
    # O mês de recebimento é apenas a data financeira do pagamento.
    # Se houver férias na própria competência, o salário mensal é
    # proporcional aos dias que não estão em gozo de férias; as
    # férias entram em evento separado.
    dias_ferias_na_competencia = 0
    if (
        dados_ferias
        and dados_ferias["eh_mes_vigente"]
    ):
        dias_ferias_na_competencia = dados_ferias["dias_gozo"]

    dias_salario_competencia = max(
        30 - dias_ferias_na_competencia,
        0
    )

    salario_competencia = arredonda(
        salario * Decimal(dias_salario_competencia) / Decimal("30")
    )

    # Verbas adicionais do mês
    adicionais_bruto = arredonda(
        valor_adicional
        + total_he
    )

    adicionais_tributaveis = arredonda(
        valor_adicional
        + total_he
    )

    # Férias são demonstradas como evento próprio.
    # Não somamos o valor das férias novamente ao salário mensal,
    # evitando duplicidade.
    ferias_competencia_bruto = Decimal("0.00")
    ferias_competencia_tributavel = Decimal("0.00")

    if (
        dados_ferias
        and dados_ferias["eh_mes_vigente"]
    ):
        ferias_competencia_bruto = dados_ferias["total_bruto"]
        ferias_competencia_tributavel = dados_ferias["total_tributavel"]

    # ------------------------------------------------------------
    # BRUTO
    # ------------------------------------------------------------

    bruto_mensal = arredonda(
        salario_competencia
        - total_desconto_faltas
        + adicionais_bruto
        + ferias_competencia_bruto
    )

    # ------------------------------------------------------------
    # BASE TRIBUTÁVEL
    # ------------------------------------------------------------

    base_tributavel = arredonda(
        salario_competencia
        - total_desconto_faltas
        + adicionais_tributaveis
        + ferias_competencia_tributavel
    )

    # ------------------------------------------------------------
    # INSS
    # ------------------------------------------------------------

    inss_mensal = calc_inss(
        base_tributavel
    )

    # ------------------------------------------------------------
    # IRRF
    # ------------------------------------------------------------

    base_irrf_mensal = arredonda(
        base_tributavel
        - inss_mensal
        - (
            Decimal(dependentes)
            * DEDUCAO_DEPENDENTE_IRRF
        )
    )

    irrf_mensal = calc_irrf(
        base_irrf_mensal,
        base_tributavel
    )

    # ------------------------------------------------------------
    # VALE TRANSPORTE
    # ------------------------------------------------------------

    limite_vt = arredonda(
        salario * Decimal("0.06")
    )

    vt = arredonda(
        min(
            valor_vt_total,
            limite_vt
        )
    )

    # ------------------------------------------------------------
    # LÍQUIDO
    # ------------------------------------------------------------

    liquido_mensal = arredonda(
        bruto_mensal
        - inss_mensal
        - irrf_mensal
        - vt
    )

    # ============================================================
    # HOLERITE
    # ============================================================

    print("\n" + "=" * 70)
    print(
        f"     HOLERITE MENSAL - COMPETÊNCIA: "
        f"{NOMES_MESES[mes_competencia].upper()}/{ano_competencia}"
    )
    print(
        f"     RECEBIMENTO DO SALÁRIO: "
        f"{NOMES_MESES[mes_pagamento_salario].upper()}/{ano_pagamento_salario}"
    )
    print("=" * 70)

    print(
        f"Salário Contratual        : {dinheiro(salario)}"
    )
    if dias_ferias_na_competencia > 0:
        print(
            f"Salário da Competência    : {dinheiro(salario_competencia)} "
            f"({dias_salario_competencia} dias fora das férias)"
        )
    else:
        print(
            f"Salário da Competência    : {dinheiro(salario_competencia)}"
        )

    if valor_adicional > 0:

        print(
            f"Adicional Risco           : "
            f"{dinheiro(valor_adicional)} "
            f"({desc_adicional})"
        )

    print(
        f"Remuneração Base         : "
        f"{dinheiro(remuneracao_base)}"
    )

    print(
        f"Hora Base ({carga_horaria:.0f}h)        : "
        f"{dinheiro(hora_base)}"
    )

    print("-" * 70)

    # ------------------------------------------------------------
    # FALTAS
    # ------------------------------------------------------------

    if dias_global > 0:

        print(
            "RESUMO DE FALTAS INFORMADAS"
        )

        for mes, dados in sorted(
            mapa_faltas.items()
        ):

            print(
                f"  -> Mês {mes:02d}: "
                f"Sem1:{dados[1]}d | "
                f"Sem2:{dados[2]}d | "
                f"Sem3:{dados[3]}d | "
                f"Sem4:{dados[4]}d "
                f"(Total: {dados['total_dias']}d)"
            )

        print(
            f"Valor do dia de trabalho : "
            f"{dinheiro(valor_dia)}"
        )

        print(
            f"Desconto Faltas "
            f"({dias_vigente}d)       : "
            f"- {dinheiro(desconto_faltas)}"
        )

        print(
            f"Desconto DSR "
            f"({semanas_vigente} sem)       : "
            f"- {dinheiro(desconto_dsr)}"
        )

        print(
            f"Total Desc. Faltas Folha Atual : "
            f"- {dinheiro(total_desconto_faltas)}"
        )

        print("-" * 70)

    # ------------------------------------------------------------
    # HORAS EXTRAS
    # ------------------------------------------------------------

    print(
        f"HE 50%  (Unit: {dinheiro(u_he50)}) "
        f"x {he50_qtd}h = {dinheiro(v_he50)}"
    )

    print(
        f"HE 70%  (Unit: {dinheiro(u_he70)}) "
        f"x {he70_qtd}h = {dinheiro(v_he70)}"
    )

    print(
        f"HE 100% (Unit: {dinheiro(u_he100)}) "
        f"x {he100_qtd}h = {dinheiro(v_he100)}"
    )

    print(
        f"Total Horas Extras       : "
        f"{dinheiro(total_he)}"
    )

    if media_variaveis > 0:
        print(
            f"Média Variáveis (12m)    : "
            f"{dinheiro(media_variaveis)}"
        )

    print("-" * 70)

    # ------------------------------------------------------------
    # FÉRIAS
    # ------------------------------------------------------------

    if dados_ferias:

        if dados_ferias["eh_mes_vigente"]:
            status = "INCLUÍDO NA FOLHA"
        else:
            status = (
                f"PROGRAMADO PARA "
                f"{dados_ferias['mes_gozo']:02d}/"
                f"{dados_ferias['ano_gozo']}"
            )

        print(
            f"FÉRIAS ({status}):"
        )

        print(
            f"  -> Valor do dia              : "
            f"{dinheiro(dados_ferias['valor_dia'])}"
        )

        print(
            f"  -> Gozo "
            f"({dados_ferias['dias_gozo']}d)           : "
            f"{dinheiro(dados_ferias['valor_ferias'])}"
        )

        print(
            f"  -> 1/3 Constitucional        : "
            f"{dinheiro(dados_ferias['terco_ferias'])}"
        )

        print(
            f"  -> Total Tributável          : "
            f"{dinheiro(dados_ferias['total_tributavel'])}"
        )

        if dados_ferias["vendeu_abono"]:

            print(
                f"  -> Abono 10 dias             : "
                f"{dinheiro(dados_ferias['valor_abono'])}"
            )

            print(
                f"  -> 1/3 sobre Abono           : "
                f"{dinheiro(dados_ferias['terco_abono'])}"
            )

            print(
                f"  -> Total Isento              : "
                f"{dinheiro(dados_ferias['total_isento'])}"
            )

        print(
            f"  -> Bruto Total Férias        : "
            f"{dinheiro(dados_ferias['total_bruto'])}"
        )

        print("-" * 70)

    # ------------------------------------------------------------
    # 13º
    # ------------------------------------------------------------

    if dados_13:

        if dados_13["eh_mes_vigente"]:
            status = "FOLHA SEPARADA DE 13º"
        else:
            status = (
                f"PROGRAMADO PARA "
                f"{dados_13['mes_pago']:02d}/"
                f"{dados_13['ano_pago']}"
            )

        print(
            f"DEMONSTRATIVO DE 13º SALÁRIO ({status}):"
        )

        print(
            f"  -> Base Fixa Proporcional "
            f"({dados_13['avos']}/12) : "
            f"{dinheiro(dados_13['base_fixa_13'])}"
        )

        print(
            f"  -> Média Variável 12m      : "
            f"{dinheiro(dados_13['media_variavel_13'])}"
        )

        print(
            f"  -> Base Bruta c/ Médias    : "
            f"{dinheiro(dados_13['base_total_13'])}"
        )

        print(
            f"  -> 1ª Parcela (50% Fixo)    : "
            f"{dinheiro(dados_13['primeira_parcela'])}"
        )

        print(
            f"  -> INSS 13º                 : "
            f"- {dinheiro(dados_13['inss_13'])}"
        )

        print(
            f"  -> Base IRRF 13º            : "
            f"{dinheiro(dados_13['base_irrf_13'])}"
        )

        print(
            f"  -> IRRF 13º                 : "
            f"- {dinheiro(dados_13['irrf_13'])}"
        )

        print(
            f"  -> 2ª Parcela LÍQUIDA       : "
            f"{dinheiro(dados_13['segunda_parcela'])}"
        )

        print("-" * 70)

    # ------------------------------------------------------------
    # RESUMO FINAL
    # ------------------------------------------------------------

    print(
        f"SALÁRIO BRUTO TOTAL MÊS : "
        f"{dinheiro(bruto_mensal)}"
    )
    print(
        f"RECEBIMENTO DO SALÁRIO  : "
        f"{NOMES_MESES[mes_pagamento_salario].upper()}/{ano_pagamento_salario}"
    )

    print(
        f"BASE TRIBUTÁVEL INSS/IRRF: "
        f"{dinheiro(base_tributavel)}"
    )

    print("-" * 70)

    print(
        f"Desconto INSS Mensal     : "
        f"- {dinheiro(inss_mensal)}"
    )

    print(
        f"Base de IRRF Mensal      : "
        f"{dinheiro(base_irrf_mensal)}"
    )

    print(
        f"Desconto IRRF Mensal     : "
        f"- {dinheiro(irrf_mensal)}"
    )


    print(
        f"Desconto VT (máx. 6%)    : "
        f"- {dinheiro(vt)}"
    )

    print("-" * 70)

    print(
        f"SALÁRIO LÍQUIDO MÊS      : "
        f"{dinheiro(liquido_mensal)}"
    )

    print("=" * 70)


# ================================================================
# MAIN
# ================================================================

# ================================================================
# ENTRADA SEGURA PARA GOOGLE COLAB
# ================================================================

if __name__ == "__main__":
    try:
        executar_calculadora()
    except ValueError as erro:
        print(f"\n⚠️ Erro de validação: {erro}")
    except Exception as erro:
        print(f"\n❌ Erro inesperado: {erro}")


        CALCULADORA FOPAG SENAI 2026
Digite o Salário Fixo (R$) [2950.00]: 
Possui adicional de Insalubridade ou Periculosidade? (s/n) [n]: 
Carga Horária Mensal em Horas (ex: 220, 200) [220.00]: 
Mês Vigente da Folha (1 a 12) [9]: 
Em qual mês/ano o salário desta competência será recebido? [10-2026]: 
----------------------------------------------------------------------
Houve faltas injustificáveis? (s/n) [n]: 
----------------------------------------------------------------------
Horas Extras 50% (quantidade) [18.00]: 
Horas Extras 70% (quantidade) [0.00]: 
Horas Extras 100% (quantidade) [10.00]: 
Número de Dependentes [1]: 
Deseja informar a média de 12 meses de verbas variáveis? (s/n) [n]: 
----------------------------------------------------------------------
Deseja programar/calcular Férias? (s/n) [n]: 
----------------------------------------------------------------------
Deseja calcular 13º Salário? (s/n) [n]: 
------------------------------------------------------------------